# 🚀 Фабрика нейросетей & Асинхронный Walk-Forward рой PRL/USDT (Google Colab / GPU T4)
Весь тяжёлый код архитектур и обучения вынесен в модульные скрипты  и .
Управляйте параметрами обучения напрямую через аргументы в ячейках ниже.

In [1]:
# 1. Инициализация окружения Colab и принудительное обновление скриптов из репозитория
!nvidia-smi
import os, sys, importlib
if os.path.exists("/content"):
    !rm -rf _repo_tmp && git clone --depth 1 https://github.com/BulletproofBoris/crypto_bot.git _repo_tmp 2>/dev/null || true
    !mkdir -p _tools && cp -rf _repo_tmp/_tools/* _tools/
sys.path.insert(0, os.path.abspath("."))
import _tools.colab_models, _tools.colab_neural_trainer
importlib.reload(_tools.colab_models)
importlib.reload(_tools.colab_neural_trainer)
from _tools.colab_neural_trainer import run_colab_pipeline
print("✅ Свежие модули _tools.colab_models и _tools.colab_neural_trainer загружены!")

Tue Oct 20 10:24:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Конфигурация и запуск Walk-Forward обучения нейросетей (редактируемые параметры)
# Асинхронная сетка окон: H=25 -> 2 окна (375k), H=12 -> 5 окон (210k), H=5 -> 12 окон (75k)
HORIZON = 5                                               # Горизонт H (сек): 5, 12 или 25
WINDOWS = "all"                                            # Список окон: "all" или "0,1"
ARCHS = "conv1d+gru,causal_cnn,attention,mlp_bottleneck"   # Архитектуры через запятую
RUNS_PER_ARCH = 15                                         # Число независимых ранов на архитектуру
MAX_EPOCHS = 10                                            # Макс. эпох на один ран
BATCH_SIZE = 2048                                          # Размер батча на GPU T4
LR = 1e-2                                                  # Скорость обучения AdamW

df_leaderboard = run_colab_pipeline(
    horizon=HORIZON, windows=WINDOWS, archs=ARCHS,
    runs_per_arch=RUNS_PER_ARCH, max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE, lr=LR
)

FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1crf-rj1bB6v0nPSuv58Qe9koCNxjEz3E

but Gdown can't. Please check connections and permissions.

In [ ]:
# 3. Альтернативный запуск через CLI с аргументами командной строки и выгрузка архива
# !python -m _tools.colab_neural_trainer --horizon 12 --windows 0,1,2,3,4 --runs 10 --epochs 10 --batch-size 2048
ZIP_NAME = f"prl_ensemble_pack_h{HORIZON}s.zip"
if os.path.exists(ZIP_NAME):
    !curl -s -F "reqtype=fileupload" -F "time=24h" -F "fileToUpload=@{ZIP_NAME}" https://litterbox.catbox.moe/resources/internals/api.php
    try:
        from google.colab import files
        files.download(ZIP_NAME)
    except Exception:
        pass